# Chapter 13 - Nonlinear Twiss

With Truncated Power Series Algebra (TPSA) tools, Twiss parameters in SciBmad can be expressed as Taylor series in selected variables. In this chapter, the selected variable is mainly the momentum deviation $\delta$.

For example, a beta function can be represented as

$$
\beta(s,\delta) = \beta_0(s) + \frac{\partial \beta}{\partial \delta}(s)\delta + \frac{1}{2!}\frac{\partial^2 \beta}{\partial \delta^2}(s)\delta^2 + \cdots.
$$

SciBmad exposes these coefficients through named Twiss columns such as `dx`, `dx_2`, `dbeta1`, and `dbeta1_2`. This is different from a finite-difference scan: the coefficients are part of the map-based calculation itself.

We begin by loading the ESR v6.3.1 lattice used by the SciBmad nonlinear Twiss example.

In [ ]:
# Activate the shared tutorial environment.
# Run setup_environment.ipynb first on a new installation or after dependencies change.
import Pkg
Pkg.activate(@__DIR__)


In [ ]:
using SciBmad
using CairoMakie
using LaTeXStrings

set_theme!(theme_latexfonts())

In [ ]:
ring = include(joinpath(pwd(), "lattices", "chapter_13", "esr-v6.3.1.jl"))

## 13.1 Initializing Twiss

A standard `twiss` call returns a `Twiss` object with a global summary in `tw.summ` and an element-by-element dataframe in `tw.df`. The columns `beta1` and `beta2` are the two transverse normal-mode beta functions in the Teng-Edwards coupling formalism. Columns can also be accessed directly as `tw.beta1`, `tw.beta2`, and so on.

In [ ]:
tw = twiss(ring)
t = tw.df

In [ ]:
fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Beta Function [m]")

lines!(ax, t.s, t.beta1, label=L"\beta_1")
lines!(ax, t.s, t.beta2, label=L"\beta_2")
axislegend(ax, position=:rt)

fig

## 13.2 TPSA Operations

The closed orbit and its momentum dependence are available as named Twiss columns. At a fixed location, the orbit can be interpreted as a Taylor series in the momentum deviation $\delta$.

$$
x_{co}(\delta) = x_0 + \eta_x\delta + \frac{1}{2!}\frac{\partial \eta_x}{\partial \delta}\delta^2 + \cdots.
$$

The columns `dx` and `dy` give the linear periodic dispersions directly. Higher Taylor coefficients use the suffix convention `dx_2`, `dy_2`, and so on.

In [ ]:
# Inspect the closed orbit at the beginning of the ring.
t.x[1]

In [ ]:
eta_x = t.dx
eta_y = t.dy

fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Dispersion [m]")

lines!(ax, t.s, eta_x, label=L"\eta_x")
lines!(ax, t.s, eta_y, label=L"\eta_y")
axislegend(ax, position=:rt)

fig

To obtain higher-order chromatic quantities, request their names with `cols` and set the required `chrom` order. Because this ring contains RF cavities, we use `rf_on=false` so that $\delta$ is a coasting-beam expansion variable. A chromatic order of 2 is enough for `dx_2` and the first beta derivatives; second-order beta coefficients require `chrom=3`.

In [ ]:
tw2 = twiss(ring; rf_on=false, chrom=2,
    cols=["x", "dx", "dx_2", "y", "dy", "dy_2",
          "beta1", "beta2", "dbeta1", "dbeta2", "w1", "w2"])
t2 = tw2.df
t2[1, [:x, :dx, :dx_2]]

The display above includes the constant orbit, its linear dispersion, and its $\delta^2$ Taylor coefficient. At the first dataframe row, which is the beginning of the ring in this lattice, `dx` is the first derivative and `dx_2` is the coefficient multiplying $\delta^2$. Therefore the second derivative at $\delta=0$ is $2!\,dx_2$.

Here `s_start` identifies the location, while the three following values are the Taylor coefficients evaluated at that location.

In [ ]:
s_start = t2.s[1]
xco_start = t2.x[1]
dxco_ddelta_start = t2.dx[1]
d2xco_ddelta2_start = 2 * t2.dx_2[1]

(s_start=s_start,
 xco_start=xco_start,
 dxco_ddelta_start=dxco_ddelta_start,
 d2xco_ddelta2_start=d2xco_ddelta2_start)

In [ ]:
# The delta^2 coefficient contains the Taylor factor 1/2!.
deta_ddelta_x = 2 .* t2.dx_2
deta_ddelta_y = 2 .* t2.dy_2

fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Second-Order Dispersion [m]")

lines!(ax, t2.s, deta_ddelta_x, label=L"\frac{\partial \eta_x}{\partial \delta}")
lines!(ax, t2.s, deta_ddelta_y, label=L"\frac{\partial \eta_y}{\partial \delta}")
axislegend(ax, position=:rt)

fig

For the closed orbit, `chrom=2` is enough to obtain both the first and second derivatives with respect to $\delta$. For beta functions, the first chromatic derivative is obtained from `t2`; the second-order coefficient is requested below as `dbeta1_2` and `dbeta2_2` with `chrom=3`.

In [ ]:
tw3 = twiss(ring; rf_on=false, chrom=3,
    cols=["beta1", "beta2", "dbeta1", "dbeta2",
          "dbeta1_2", "dbeta2_2"])
t3 = tw3.df;

The named beta columns follow the same convention. Here we use the design beta, the first derivative with respect to $\delta$, and twice the second-order Taylor coefficient to obtain the second derivative at $\delta=0$.

For inspection, `t2.beta1[1]` shows the design beta at the first lattice position.

In [ ]:
t2.beta1[1]

In [ ]:
beta1 = t2.beta1
beta2 = t2.beta2

dbeta_ddelta_1 = t2.dbeta1
dbeta_ddelta_2 = t2.dbeta2

d2beta_ddelta2_1 = 2 .* t3.dbeta1_2
d2beta_ddelta2_2 = 2 .* t3.dbeta2_2

nothing

In [ ]:
fig = Figure(fontsize=22, size=(900, 650))

ax1 = Axis(fig[1, 1], xlabel="s [m]", ylabel=L"\partial\beta/\partial\delta \; [m]")
lines!(ax1, t2.s, dbeta_ddelta_1, label=L"\beta_1")
lines!(ax1, t2.s, dbeta_ddelta_2, label=L"\beta_2")
axislegend(ax1, position=:rt)

ax2 = Axis(fig[2, 1], xlabel="s [m]", ylabel=L"\partial^2\beta/\partial\delta^2 \; [m]")
lines!(ax2, t3.s, d2beta_ddelta2_1, label=L"\beta_1")
lines!(ax2, t3.s, d2beta_ddelta2_2, label=L"\beta_2")
axislegend(ax2, position=:rt)

fig

## 13.3 Examples: Chromatic Beats and Montague Functions

A common way to show the chromatic distortion of the beta function is the normalized beta derivative

$$
\frac{1}{\beta}\frac{\partial \beta}{\partial \delta}.
$$

This is often called chromatic beta-beat.

In [ ]:
fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Chromatic Beta-Beat")

lines!(ax, t2.s, dbeta_ddelta_1 ./ beta1, label=L"\frac{\partial\beta_1/\partial\delta}{\beta_1}")
lines!(ax, t2.s, dbeta_ddelta_2 ./ beta2, label=L"\frac{\partial\beta_2/\partial\delta}{\beta_2}")
axislegend(ax, position=:rt)

fig

The Montague functions combine the chromatic distortion of both $\alpha$ and $\beta$:

$$
W = \sqrt{A^2 + B^2},
$$

$$
A = \frac{\partial\alpha}{\partial\delta} - \frac{\alpha}{\beta}\frac{\partial\beta}{\partial\delta}, \qquad
B = \frac{1}{\beta}\frac{\partial\beta}{\partial\delta}.
$$

In achromatic regions, these functions are useful phase-invariant diagnostics of chromatic optical distortion. SciBmad computes them directly with the `w1` and `w2` columns, so no manual reconstruction from alpha and beta columns is needed.

In [ ]:
w1 = t2.w1
w2 = t2.w2

# plot figure
fig = Figure(fontsize=22, size=(700, 360))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Montague Functions")

lines!(ax, t2.s, w1, label=L"W_1")
lines!(ax, t2.s, w2, label=L"W_2")
axislegend(ax, position=:lt)

fig

The requested derivative columns are ordinary dataframe columns. This makes it easy to inspect or export selected nonlinear optics quantities without handling the underlying TPSA representation directly.

In [ ]:
t3[1:5, [:s, :dbeta1, :dbeta2, :dbeta1_2, :dbeta2_2]]

## 13.4 Tunes

The global tunes are returned in the Twiss summary as `q1` and `q2`. For a nonlinear calculation they are `AmplitudeDependentValue` objects indexed by named powers such as `J1`, `J2`, and, for a coasting beam, `delta`. In the phasor basis, the transverse variables are written in terms of $\sqrt{J}e^{\pm i\phi}$.

For example, a monomial with exponents `1 1` in one transverse mode corresponds to

$$
(\sqrt{J}e^{-i\phi})(\sqrt{J}e^{+i\phi}) = J.
$$

Therefore, `q1[J1=1]` and `q1[J2=1]` give amplitude-dependent tune coefficients for the two normal-mode actions.

In [ ]:
tw_nonlinear = twiss(ring; rf_on=false, order=3, chrom=3, at=[])

(q1=tw_nonlinear.q1,
 dq1_dJ1=tw_nonlinear.q1[J1=1],
 dq1_dJ2=tw_nonlinear.q1[J2=1])

To plot tune variation with momentum deviation, compute the summary to higher chromatic order and extract its coefficients with `q1[delta=n]` and `q2[delta=n]`. These indexed values are Taylor coefficients, not $n$th derivatives; the derivative at the origin is $n!$ times the coefficient. The keyword `at=[]` keeps only the mandatory start/end rows while the global quantities are computed.

In [ ]:
tw_tunes = twiss(ring; rf_on=false, order=1, chrom=6, at=[])
delta = (-1:0.01:1) .* 1e-2

q1_coeffs = [tw_tunes.q1[delta=n] for n in 0:6]
q2_coeffs = [tw_tunes.q2[delta=n] for n in 0:6]
Qx = evalpoly.(delta, Ref(q1_coeffs))
Qy = evalpoly.(delta, Ref(q2_coeffs))

fig = Figure(fontsize=22, size=(640, 500))
ax = Axis(fig[1, 1], xlabel=L"\delta", ylabel="Fractional Tunes")

lines!(ax, delta, Qx, label=L"Q_x")
lines!(ax, delta, Qy, label=L"Q_y")
axislegend(ax, position=:rt)

fig

## 13.5 Analyzing Spin

SciBmad can also include spin in the nonlinear Twiss calculation. Setting `spin=true` and requesting `nx`, `ny`, `nz`, `dnx`, `dny`, and `dnz` adds the stable spin direction $\hat n$ and its momentum derivatives to the Twiss dataframe. The derivative

$$
\frac{\partial \hat n}{\partial \delta}
$$

measures how sensitively the spin direction changes with momentum deviation.

In [ ]:
tw_spin = twiss(ring; spin=true,
    cols=["nx", "ny", "nz", "dnx", "dny", "dnz"])

In [ ]:
ts = tw_spin.df
dn_ddelta_x = ts.dnx
dn_ddelta_y = ts.dny
dn_ddelta_z = ts.dnz
dn_ddelta_amp = sqrt.(vec(sum(abs2, [dn_ddelta_x dn_ddelta_y dn_ddelta_z], dims=2)))

fig = Figure(fontsize=22, size=(760, 380))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Spin-Orbit Coupling")

lines!(ax, ts.s, dn_ddelta_x, label=L"\frac{\partial n_x}{\partial\delta}")
lines!(ax, ts.s, dn_ddelta_y, label=L"\frac{\partial n_y}{\partial\delta}")
lines!(ax, ts.s, dn_ddelta_z, label=L"\frac{\partial n_z}{\partial\delta}")
lines!(ax, ts.s, dn_ddelta_amp, label=L"\left|\frac{\partial\hat{n}}{\partial\delta}\right|")
Legend(fig[1, 2], ax)

fig

## 13.6 Exercises

1. **Derivative order check.** Use `chrom=1`, `chrom=2`, and `chrom=3` with appropriate `cols` to request `dx`, `dx_2`, `dbeta1`, and `dbeta1_2`. Explain why the beta-function coefficients require one higher chromatic order than the corresponding closed-orbit coefficients. Remember to set `rf_on=false`.

2. **Optics sensitivity.** Find the location where $|\partial\beta_x/\partial\delta|/\beta_x$ is largest. What element or region of the ring is closest to this maximum? Repeat the same check for the vertical plane.

3. **Spin sensitivity.** Find the location where $|\partial\hat{n}/\partial\delta|$ is largest. Compare this location with the strongest chromatic beta-beating region. Are they near the same part of the ring? What might this suggest about the difference between optics sensitivity and spin sensitivity?

4. **Tune coefficients.** Extract `q1[delta=n]` and `q2[delta=n]` for several orders. Convert each Taylor coefficient to the corresponding derivative at $\delta=0$, and verify the factorial relationship numerically.